In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Results Fact
# MAGIC
# MAGIC 1. Read Silver `results` data
# MAGIC 2. Read Silver `sprints` data
# MAGIC 3. Add `session_type` as `RACE` or `SPRINT`
# MAGIC 4. UNION `results` and `sprints`
# MAGIC 5. Derive `is_win`, `is_podium` and `has_points`
# MAGIC 6. Write the transformed data to Gold `fact_session_results`

# COMMAND ----------

# MAGIC %run "../00.common/01.environment-config.py"

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

results_path = f"{s3_root_path}/Silver/Nisarga/results"

sprints_path = f"{s3_root_path}/Silver/Nisarga/sprints"

gold_path = f"{s3_root_path}/Gold/Nisarga/fact_session_results"

print("Results Path :", results_path)
print("Sprints Path :", sprints_path)
print("Gold Path    :", gold_path)

# COMMAND ----------

# ============================================================
# STEP 1 - READ SILVER RESULTS DATA
# ============================================================

results_df = (
    spark.read
    .format("delta")
    .load(results_path)
)

display(results_df)

# COMMAND ----------

# ============================================================
# STEP 2 - READ SILVER SPRINTS DATA
# ============================================================

sprints_df = (
    spark.read
    .format("delta")
    .load(sprints_path)
)

display(sprints_df)

# COMMAND ----------

# ============================================================
# STEP 3 - ADD SESSION TYPE
# ============================================================

results_session_df = (
    results_df
    .withColumnRenamed("date", "race_date")
    .withColumnRenamed("grid", "grid_position")
    .withColumnRenamed("laps", "completed_laps")
    .withColumnRenamed("number", "car_number")
    .withColumnRenamed("position", "final_position")
    .withColumnRenamed("position_text", "final_position_text")
    .withColumn("session_type", F.lit("RACE"))
    .drop(
        "ingestion_timestamp",
        "source_file",
        "is_winner"
    )
)

sprints_session_df = (
    sprints_df
    .withColumn("session_type", F.lit("SPRINT"))
    .drop(
        "ingestion_timestamp",
        "source_file"
    )
)

display(results_session_df)
display(sprints_session_df)

# COMMAND ----------

# ============================================================
# STEP 4 - UNION RESULTS AND SPRINTS
# ============================================================

results_sprints_df = (
    results_session_df
    .unionByName(sprints_session_df)
)

display(results_sprints_df)

# COMMAND ----------

# ============================================================
# STEP 5 - DERIVE BUSINESS FLAGS
# ============================================================

fact_session_results_df = (
    results_sprints_df
    .withColumn(
        "is_win",
        F.col("final_position") == 1
    )
    .withColumn(
        "is_podium",
        F.col("final_position").between(1, 3)
    )
    .withColumn(
        "has_points",
        F.col("points") > 0
    )
)

display(fact_session_results_df)

# COMMAND ----------

# ============================================================
# STEP 6 - WRITE TO GOLD
# ============================================================

(
    fact_session_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)

print("Gold Fact Session Results successfully written to:")
print(gold_path)

# COMMAND ----------

# ============================================================
# STEP 7 - VALIDATE GOLD DATA
# ============================================================

gold_fact_session_results_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

display(gold_fact_session_results_df)

print("Silver Results Records :", results_df.count())
print("Silver Sprints Records :", sprints_df.count())
print("Gold Records           :", gold_fact_session_results_df.count())

# COMMAND ----------

# ============================================================
# STEP 8 - CHECK SESSION TYPE
# ============================================================

display(
    gold_fact_session_results_df
    .groupBy("session_type")
    .count()
    .orderBy("session_type")
)

# COMMAND ----------

# ============================================================
# STEP 9 - CHECK WIN / PODIUM / POINTS
# ============================================================

display(
    gold_fact_session_results_df
    .groupBy("session_type")
    .agg(
        F.sum(F.col("is_win").cast("int")).alias("wins"),
        F.sum(F.col("is_podium").cast("int")).alias("podiums"),
        F.sum(F.col("has_points").cast("int")).alias("drivers_with_points")
    )
    .orderBy("session_type")
)